# ZGLMM159 — M2 — Suprimento e parâmetros de estoque — Cenário Centro 4058

**View:** `dev_procurement.corp_curated.vw_zglmm159_m2`  
**Transação SAP:** ZGLMM159  
**Cenário:** `cod_centro = '4058'`  
**Objetivo:** repetir no novo desenho C1/M1/M2 o mesmo cenário validado anteriormente para a ZGLMM159, preservando a granularidade específica de cada visão.  
**Colunas no DESCRIBE recebido:** 28

## Como usar
Execute **Run All** no Databricks. As consultas leem a view diretamente e não criam tabelas ou views temporárias.

## Atenções iniciais
- A chave abaixo é apenas uma **candidata** e será confirmada pelos testes de unicidade.
- Colunas comentadas como sem fonte ou `NULL` serão medidas, não presumidas.
- Volumetria igual não comprova qualidade dos dados.

**Referência histórica do cenário:** centro 4058. Na validação anterior, C1 e M2 apresentaram granularidade de uma linha por material, enquanto M1 apresentou granularidade de material + depósito. Os testes abaixo reconfirmam esse comportamento nas novas views.


## 1. Metadados e definição da view


In [0]:
DESCRIBE EXTENDED dev_procurement.corp_curated.vw_zglmm159_m2;


In [0]:
SHOW CREATE TABLE dev_procurement.corp_curated.vw_zglmm159_m2;


## Preparação do cenário — centro 4058

A view temporária abaixo garante que todas as análises usem exatamente o mesmo recorte. A contagem de controle confirma que nenhum outro centro entrou no cenário.


In [0]:
CREATE OR REPLACE TEMP VIEW tmp_zglmm159_m2_centro_4058 AS
SELECT *
FROM dev_procurement.corp_curated.vw_zglmm159_m2
WHERE TRIM(CAST(cod_centro AS STRING)) = '4058';


In [0]:
SELECT
  COUNT(*) AS total_linhas_cenario,
  COUNT(DISTINCT cod_centro) AS centros_distintos,
  MIN(cod_centro) AS centro_minimo,
  MAX(cod_centro) AS centro_maximo
FROM tmp_zglmm159_m2_centro_4058;


## 2. Volumetria geral


In [0]:
SELECT
  COUNT(*) AS total_linhas,
  COUNT(DISTINCT cod_material) AS materiais_distintos,
  COUNT(DISTINCT cod_centro) AS centros_distintos,
  COUNT(DISTINCT cod_empresa) AS empresas_distintas
  
FROM tmp_zglmm159_m2_centro_4058;


## 3. Distribuição por centro e empresa — base para definição de cenários


In [0]:
SELECT
  cod_centro,
  cod_empresa,
  COUNT(*) AS linhas,
  COUNT(DISTINCT cod_material) AS materiais,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct_linhas
FROM tmp_zglmm159_m2_centro_4058
GROUP BY cod_centro, cod_empresa
ORDER BY linhas DESC, cod_centro, cod_empresa;


## 4. Granularidade e testes de chave candidata


In [0]:
SELECT 'cod_material' AS chave, COUNT(*) AS linhas, COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'))) AS distintos,
ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'))), 0), 6) AS linhas_por_chave
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'cod_material + cod_centro' AS chave, COUNT(*) AS linhas, COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_centro` AS STRING)), '<NULL>'))) AS distintos,
ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_centro` AS STRING)), '<NULL>'))), 0), 6) AS linhas_por_chave
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'cod_material + cod_centro + cod_empresa' AS chave, COUNT(*) AS linhas, COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_centro` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_empresa` AS STRING)), '<NULL>'))) AS distintos,
ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_centro` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_empresa` AS STRING)), '<NULL>'))), 0), 6) AS linhas_por_chave
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'cod_material + cod_centro + cod_empresa + tp_suprimento_especial' AS chave, COUNT(*) AS linhas, COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_centro` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_empresa` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`tp_suprimento_especial` AS STRING)), '<NULL>'))) AS distintos,
ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_centro` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_empresa` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`tp_suprimento_especial` AS STRING)), '<NULL>'))), 0), 6) AS linhas_por_chave
FROM tmp_zglmm159_m2_centro_4058;


### Duplicidades na chave candidata: `cod_material + cod_centro + cod_empresa`


In [0]:
SELECT `cod_material`, `cod_centro`, `cod_empresa`, COUNT(*) AS qtd
FROM tmp_zglmm159_m2_centro_4058
GROUP BY `cod_material`, `cod_centro`, `cod_empresa`
HAVING COUNT(*) > 1
ORDER BY qtd DESC, `cod_material`, `cod_centro`, `cod_empresa`;


### Inspeção campo a campo das chaves repetidas


In [0]:
WITH duplicadas AS (
  SELECT `cod_material`, `cod_centro`, `cod_empresa`
  FROM tmp_zglmm159_m2_centro_4058
  GROUP BY `cod_material`, `cod_centro`, `cod_empresa`
  HAVING COUNT(*) > 1
)
SELECT t.*
FROM tmp_zglmm159_m2_centro_4058 t
INNER JOIN duplicadas d ON COALESCE(CAST(t.`cod_material` AS STRING), '<NULL>') = COALESCE(CAST(d.`cod_material` AS STRING), '<NULL>') AND COALESCE(CAST(t.`cod_centro` AS STRING), '<NULL>') = COALESCE(CAST(d.`cod_centro` AS STRING), '<NULL>') AND COALESCE(CAST(t.`cod_empresa` AS STRING), '<NULL>') = COALESCE(CAST(d.`cod_empresa` AS STRING), '<NULL>')
ORDER BY t.`cod_material`, t.`cod_centro`, t.`cod_empresa`;


## 5. Preenchimento de todas as colunas — nulo, zero e útil


In [0]:
SELECT 'cod_material' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `cod_material` IS NULL OR LOWER(TRIM(CAST(`cod_material` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`cod_material` IS NULL OR LOWER(TRIM(CAST(`cod_material` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`cod_material` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`cod_material` IS NULL OR LOWER(TRIM(CAST(`cod_material` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`cod_material` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `cod_material` IS NULL OR LOWER(TRIM(CAST(`cod_material` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`cod_material` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'cod_centro' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `cod_centro` IS NULL OR LOWER(TRIM(CAST(`cod_centro` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`cod_centro` IS NULL OR LOWER(TRIM(CAST(`cod_centro` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`cod_centro` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`cod_centro` IS NULL OR LOWER(TRIM(CAST(`cod_centro` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`cod_centro` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `cod_centro` IS NULL OR LOWER(TRIM(CAST(`cod_centro` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`cod_centro` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'cod_empresa' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `cod_empresa` IS NULL OR LOWER(TRIM(CAST(`cod_empresa` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`cod_empresa` IS NULL OR LOWER(TRIM(CAST(`cod_empresa` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`cod_empresa` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`cod_empresa` IS NULL OR LOWER(TRIM(CAST(`cod_empresa` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`cod_empresa` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `cod_empresa` IS NULL OR LOWER(TRIM(CAST(`cod_empresa` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`cod_empresa` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'tp_suprimento' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `tp_suprimento` IS NULL OR LOWER(TRIM(CAST(`tp_suprimento` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`tp_suprimento` IS NULL OR LOWER(TRIM(CAST(`tp_suprimento` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`tp_suprimento` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`tp_suprimento` IS NULL OR LOWER(TRIM(CAST(`tp_suprimento` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`tp_suprimento` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `tp_suprimento` IS NULL OR LOWER(TRIM(CAST(`tp_suprimento` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`tp_suprimento` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'tp_grupo_determinacao_estoque' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `tp_grupo_determinacao_estoque` IS NULL OR LOWER(TRIM(CAST(`tp_grupo_determinacao_estoque` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`tp_grupo_determinacao_estoque` IS NULL OR LOWER(TRIM(CAST(`tp_grupo_determinacao_estoque` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`tp_grupo_determinacao_estoque` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`tp_grupo_determinacao_estoque` IS NULL OR LOWER(TRIM(CAST(`tp_grupo_determinacao_estoque` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`tp_grupo_determinacao_estoque` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `tp_grupo_determinacao_estoque` IS NULL OR LOWER(TRIM(CAST(`tp_grupo_determinacao_estoque` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`tp_grupo_determinacao_estoque` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'ind_remessa_jit' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `ind_remessa_jit` IS NULL OR LOWER(TRIM(CAST(`ind_remessa_jit` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`ind_remessa_jit` IS NULL OR LOWER(TRIM(CAST(`ind_remessa_jit` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`ind_remessa_jit` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`ind_remessa_jit` IS NULL OR LOWER(TRIM(CAST(`ind_remessa_jit` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`ind_remessa_jit` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `ind_remessa_jit` IS NULL OR LOWER(TRIM(CAST(`ind_remessa_jit` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`ind_remessa_jit` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'cod_chave_horizonte' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `cod_chave_horizonte` IS NULL OR LOWER(TRIM(CAST(`cod_chave_horizonte` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`cod_chave_horizonte` IS NULL OR LOWER(TRIM(CAST(`cod_chave_horizonte` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`cod_chave_horizonte` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`cod_chave_horizonte` IS NULL OR LOWER(TRIM(CAST(`cod_chave_horizonte` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`cod_chave_horizonte` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `cod_chave_horizonte` IS NULL OR LOWER(TRIM(CAST(`cod_chave_horizonte` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`cod_chave_horizonte` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'ind_entrada_lotes' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `ind_entrada_lotes` IS NULL OR LOWER(TRIM(CAST(`ind_entrada_lotes` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`ind_entrada_lotes` IS NULL OR LOWER(TRIM(CAST(`ind_entrada_lotes` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`ind_entrada_lotes` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`ind_entrada_lotes` IS NULL OR LOWER(TRIM(CAST(`ind_entrada_lotes` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`ind_entrada_lotes` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `ind_entrada_lotes` IS NULL OR LOWER(TRIM(CAST(`ind_entrada_lotes` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`ind_entrada_lotes` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'ind_coproduto' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `ind_coproduto` IS NULL OR LOWER(TRIM(CAST(`ind_coproduto` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`ind_coproduto` IS NULL OR LOWER(TRIM(CAST(`ind_coproduto` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`ind_coproduto` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`ind_coproduto` IS NULL OR LOWER(TRIM(CAST(`ind_coproduto` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`ind_coproduto` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `ind_coproduto` IS NULL OR LOWER(TRIM(CAST(`ind_coproduto` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`ind_coproduto` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'cod_deposito_suprimento_externo' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `cod_deposito_suprimento_externo` IS NULL OR LOWER(TRIM(CAST(`cod_deposito_suprimento_externo` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`cod_deposito_suprimento_externo` IS NULL OR LOWER(TRIM(CAST(`cod_deposito_suprimento_externo` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`cod_deposito_suprimento_externo` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`cod_deposito_suprimento_externo` IS NULL OR LOWER(TRIM(CAST(`cod_deposito_suprimento_externo` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`cod_deposito_suprimento_externo` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `cod_deposito_suprimento_externo` IS NULL OR LOWER(TRIM(CAST(`cod_deposito_suprimento_externo` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`cod_deposito_suprimento_externo` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'cod_deposito_producao' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `cod_deposito_producao` IS NULL OR LOWER(TRIM(CAST(`cod_deposito_producao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`cod_deposito_producao` IS NULL OR LOWER(TRIM(CAST(`cod_deposito_producao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`cod_deposito_producao` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`cod_deposito_producao` IS NULL OR LOWER(TRIM(CAST(`cod_deposito_producao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`cod_deposito_producao` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `cod_deposito_producao` IS NULL OR LOWER(TRIM(CAST(`cod_deposito_producao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`cod_deposito_producao` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'cod_calendario_planejamento' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `cod_calendario_planejamento` IS NULL OR LOWER(TRIM(CAST(`cod_calendario_planejamento` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`cod_calendario_planejamento` IS NULL OR LOWER(TRIM(CAST(`cod_calendario_planejamento` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`cod_calendario_planejamento` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`cod_calendario_planejamento` IS NULL OR LOWER(TRIM(CAST(`cod_calendario_planejamento` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`cod_calendario_planejamento` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `cod_calendario_planejamento` IS NULL OR LOWER(TRIM(CAST(`cod_calendario_planejamento` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`cod_calendario_planejamento` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'ind_baixa_explosao' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `ind_baixa_explosao` IS NULL OR LOWER(TRIM(CAST(`ind_baixa_explosao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`ind_baixa_explosao` IS NULL OR LOWER(TRIM(CAST(`ind_baixa_explosao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`ind_baixa_explosao` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`ind_baixa_explosao` IS NULL OR LOWER(TRIM(CAST(`ind_baixa_explosao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`ind_baixa_explosao` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `ind_baixa_explosao` IS NULL OR LOWER(TRIM(CAST(`ind_baixa_explosao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`ind_baixa_explosao` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'cod_perfil_cobertura' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `cod_perfil_cobertura` IS NULL OR LOWER(TRIM(CAST(`cod_perfil_cobertura` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`cod_perfil_cobertura` IS NULL OR LOWER(TRIM(CAST(`cod_perfil_cobertura` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`cod_perfil_cobertura` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`cod_perfil_cobertura` IS NULL OR LOWER(TRIM(CAST(`cod_perfil_cobertura` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`cod_perfil_cobertura` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `cod_perfil_cobertura` IS NULL OR LOWER(TRIM(CAST(`cod_perfil_cobertura` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`cod_perfil_cobertura` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'ind_material_granel' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `ind_material_granel` IS NULL OR LOWER(TRIM(CAST(`ind_material_granel` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`ind_material_granel` IS NULL OR LOWER(TRIM(CAST(`ind_material_granel` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`ind_material_granel` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`ind_material_granel` IS NULL OR LOWER(TRIM(CAST(`ind_material_granel` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`ind_material_granel` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `ind_material_granel` IS NULL OR LOWER(TRIM(CAST(`ind_material_granel` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`ind_material_granel` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'cod_margem_seguranca' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `cod_margem_seguranca` IS NULL OR LOWER(TRIM(CAST(`cod_margem_seguranca` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`cod_margem_seguranca` IS NULL OR LOWER(TRIM(CAST(`cod_margem_seguranca` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`cod_margem_seguranca` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`cod_margem_seguranca` IS NULL OR LOWER(TRIM(CAST(`cod_margem_seguranca` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`cod_margem_seguranca` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `cod_margem_seguranca` IS NULL OR LOWER(TRIM(CAST(`cod_margem_seguranca` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`cod_margem_seguranca` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'cod_perfil_periodo' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `cod_perfil_periodo` IS NULL OR LOWER(TRIM(CAST(`cod_perfil_periodo` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`cod_perfil_periodo` IS NULL OR LOWER(TRIM(CAST(`cod_perfil_periodo` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`cod_perfil_periodo` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`cod_perfil_periodo` IS NULL OR LOWER(TRIM(CAST(`cod_perfil_periodo` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`cod_perfil_periodo` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `cod_perfil_periodo` IS NULL OR LOWER(TRIM(CAST(`cod_perfil_periodo` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`cod_perfil_periodo` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'qt_margem_seguranca' AS coluna, 'decimal(5,0)' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `qt_margem_seguranca` IS NULL OR LOWER(TRIM(CAST(`qt_margem_seguranca` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`qt_margem_seguranca` IS NULL OR LOWER(TRIM(CAST(`qt_margem_seguranca` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`qt_margem_seguranca` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`qt_margem_seguranca` IS NULL OR LOWER(TRIM(CAST(`qt_margem_seguranca` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`qt_margem_seguranca` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `qt_margem_seguranca` IS NULL OR LOWER(TRIM(CAST(`qt_margem_seguranca` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`qt_margem_seguranca` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'tp_suprimento_especial' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `tp_suprimento_especial` IS NULL OR LOWER(TRIM(CAST(`tp_suprimento_especial` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`tp_suprimento_especial` IS NULL OR LOWER(TRIM(CAST(`tp_suprimento_especial` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`tp_suprimento_especial` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`tp_suprimento_especial` IS NULL OR LOWER(TRIM(CAST(`tp_suprimento_especial` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`tp_suprimento_especial` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `tp_suprimento_especial` IS NULL OR LOWER(TRIM(CAST(`tp_suprimento_especial` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`tp_suprimento_especial` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'ind_utilizacao_quotizacao' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `ind_utilizacao_quotizacao` IS NULL OR LOWER(TRIM(CAST(`ind_utilizacao_quotizacao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`ind_utilizacao_quotizacao` IS NULL OR LOWER(TRIM(CAST(`ind_utilizacao_quotizacao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`ind_utilizacao_quotizacao` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`ind_utilizacao_quotizacao` IS NULL OR LOWER(TRIM(CAST(`ind_utilizacao_quotizacao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`ind_utilizacao_quotizacao` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `ind_utilizacao_quotizacao` IS NULL OR LOWER(TRIM(CAST(`ind_utilizacao_quotizacao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`ind_utilizacao_quotizacao` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'ind_aap' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `ind_aap` IS NULL OR LOWER(TRIM(CAST(`ind_aap` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`ind_aap` IS NULL OR LOWER(TRIM(CAST(`ind_aap` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`ind_aap` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`ind_aap` IS NULL OR LOWER(TRIM(CAST(`ind_aap` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`ind_aap` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `ind_aap` IS NULL OR LOWER(TRIM(CAST(`ind_aap` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`ind_aap` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'qt_tempo_producao_interna' AS coluna, 'decimal(5,0)' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `qt_tempo_producao_interna` IS NULL OR LOWER(TRIM(CAST(`qt_tempo_producao_interna` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`qt_tempo_producao_interna` IS NULL OR LOWER(TRIM(CAST(`qt_tempo_producao_interna` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`qt_tempo_producao_interna` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`qt_tempo_producao_interna` IS NULL OR LOWER(TRIM(CAST(`qt_tempo_producao_interna` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`qt_tempo_producao_interna` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `qt_tempo_producao_interna` IS NULL OR LOWER(TRIM(CAST(`qt_tempo_producao_interna` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`qt_tempo_producao_interna` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'qt_estoque_seguranca' AS coluna, 'decimal(18,3)' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `qt_estoque_seguranca` IS NULL OR LOWER(TRIM(CAST(`qt_estoque_seguranca` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`qt_estoque_seguranca` IS NULL OR LOWER(TRIM(CAST(`qt_estoque_seguranca` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`qt_estoque_seguranca` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`qt_estoque_seguranca` IS NULL OR LOWER(TRIM(CAST(`qt_estoque_seguranca` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`qt_estoque_seguranca` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `qt_estoque_seguranca` IS NULL OR LOWER(TRIM(CAST(`qt_estoque_seguranca` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`qt_estoque_seguranca` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'qt_estoque_minimo_seguranca' AS coluna, 'decimal(18,3)' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `qt_estoque_minimo_seguranca` IS NULL OR LOWER(TRIM(CAST(`qt_estoque_minimo_seguranca` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`qt_estoque_minimo_seguranca` IS NULL OR LOWER(TRIM(CAST(`qt_estoque_minimo_seguranca` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`qt_estoque_minimo_seguranca` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`qt_estoque_minimo_seguranca` IS NULL OR LOWER(TRIM(CAST(`qt_estoque_minimo_seguranca` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`qt_estoque_minimo_seguranca` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `qt_estoque_minimo_seguranca` IS NULL OR LOWER(TRIM(CAST(`qt_estoque_minimo_seguranca` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`qt_estoque_minimo_seguranca` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'pct_grau_atendimento' AS coluna, 'decimal(5,2)' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `pct_grau_atendimento` IS NULL OR LOWER(TRIM(CAST(`pct_grau_atendimento` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`pct_grau_atendimento` IS NULL OR LOWER(TRIM(CAST(`pct_grau_atendimento` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`pct_grau_atendimento` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`pct_grau_atendimento` IS NULL OR LOWER(TRIM(CAST(`pct_grau_atendimento` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`pct_grau_atendimento` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `pct_grau_atendimento` IS NULL OR LOWER(TRIM(CAST(`pct_grau_atendimento` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`pct_grau_atendimento` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'qt_dias_prazo_entrega_previsto' AS coluna, 'decimal(5,0)' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `qt_dias_prazo_entrega_previsto` IS NULL OR LOWER(TRIM(CAST(`qt_dias_prazo_entrega_previsto` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`qt_dias_prazo_entrega_previsto` IS NULL OR LOWER(TRIM(CAST(`qt_dias_prazo_entrega_previsto` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`qt_dias_prazo_entrega_previsto` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`qt_dias_prazo_entrega_previsto` IS NULL OR LOWER(TRIM(CAST(`qt_dias_prazo_entrega_previsto` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`qt_dias_prazo_entrega_previsto` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `qt_dias_prazo_entrega_previsto` IS NULL OR LOWER(TRIM(CAST(`qt_dias_prazo_entrega_previsto` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`qt_dias_prazo_entrega_previsto` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'qt_dias_processamento_entrada' AS coluna, 'decimal(5,0)' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `qt_dias_processamento_entrada` IS NULL OR LOWER(TRIM(CAST(`qt_dias_processamento_entrada` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`qt_dias_processamento_entrada` IS NULL OR LOWER(TRIM(CAST(`qt_dias_processamento_entrada` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`qt_dias_processamento_entrada` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`qt_dias_processamento_entrada` IS NULL OR LOWER(TRIM(CAST(`qt_dias_processamento_entrada` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`qt_dias_processamento_entrada` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `qt_dias_processamento_entrada` IS NULL OR LOWER(TRIM(CAST(`qt_dias_processamento_entrada` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`qt_dias_processamento_entrada` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'ind_marcado_exclusao' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `ind_marcado_exclusao` IS NULL OR LOWER(TRIM(CAST(`ind_marcado_exclusao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`ind_marcado_exclusao` IS NULL OR LOWER(TRIM(CAST(`ind_marcado_exclusao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`ind_marcado_exclusao` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`ind_marcado_exclusao` IS NULL OR LOWER(TRIM(CAST(`ind_marcado_exclusao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`ind_marcado_exclusao` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `ind_marcado_exclusao` IS NULL OR LOWER(TRIM(CAST(`ind_marcado_exclusao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`ind_marcado_exclusao` AS STRING)) AS distintos
FROM tmp_zglmm159_m2_centro_4058
ORDER BY pct_nulos DESC, coluna;


## 6. Colunas documentadas como sem fonte ou stub


In [0]:
SELECT
  SUM(CASE WHEN NOT (`ind_remessa_jit` IS NULL OR LOWER(TRIM(CAST(`ind_remessa_jit` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) THEN 1 ELSE 0 END) AS `ind_remessa_jit_preenchidos`,
  SUM(CASE WHEN NOT (`cod_chave_horizonte` IS NULL OR LOWER(TRIM(CAST(`cod_chave_horizonte` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) THEN 1 ELSE 0 END) AS `cod_chave_horizonte_preenchidos`,
  SUM(CASE WHEN NOT (`ind_entrada_lotes` IS NULL OR LOWER(TRIM(CAST(`ind_entrada_lotes` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) THEN 1 ELSE 0 END) AS `ind_entrada_lotes_preenchidos`,
  SUM(CASE WHEN NOT (`cod_calendario_planejamento` IS NULL OR LOWER(TRIM(CAST(`cod_calendario_planejamento` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) THEN 1 ELSE 0 END) AS `cod_calendario_planejamento_preenchidos`,
  SUM(CASE WHEN NOT (`ind_baixa_explosao` IS NULL OR LOWER(TRIM(CAST(`ind_baixa_explosao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) THEN 1 ELSE 0 END) AS `ind_baixa_explosao_preenchidos`,
  SUM(CASE WHEN NOT (`cod_perfil_cobertura` IS NULL OR LOWER(TRIM(CAST(`cod_perfil_cobertura` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) THEN 1 ELSE 0 END) AS `cod_perfil_cobertura_preenchidos`,
  SUM(CASE WHEN NOT (`cod_margem_seguranca` IS NULL OR LOWER(TRIM(CAST(`cod_margem_seguranca` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) THEN 1 ELSE 0 END) AS `cod_margem_seguranca_preenchidos`,
  SUM(CASE WHEN NOT (`cod_perfil_periodo` IS NULL OR LOWER(TRIM(CAST(`cod_perfil_periodo` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) THEN 1 ELSE 0 END) AS `cod_perfil_periodo_preenchidos`,
  SUM(CASE WHEN NOT (`qt_margem_seguranca` IS NULL OR LOWER(TRIM(CAST(`qt_margem_seguranca` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) THEN 1 ELSE 0 END) AS `qt_margem_seguranca_preenchidos`,
  SUM(CASE WHEN NOT (`ind_utilizacao_quotizacao` IS NULL OR LOWER(TRIM(CAST(`ind_utilizacao_quotizacao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) THEN 1 ELSE 0 END) AS `ind_utilizacao_quotizacao_preenchidos`,
  SUM(CASE WHEN NOT (`ind_aap` IS NULL OR LOWER(TRIM(CAST(`ind_aap` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) THEN 1 ELSE 0 END) AS `ind_aap_preenchidos`,
  SUM(CASE WHEN NOT (`qt_estoque_minimo_seguranca` IS NULL OR LOWER(TRIM(CAST(`qt_estoque_minimo_seguranca` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) THEN 1 ELSE 0 END) AS `qt_estoque_minimo_seguranca_preenchidos`,
  SUM(CASE WHEN NOT (`pct_grau_atendimento` IS NULL OR LOWER(TRIM(CAST(`pct_grau_atendimento` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) THEN 1 ELSE 0 END) AS `pct_grau_atendimento_preenchidos`
FROM tmp_zglmm159_m2_centro_4058;


## 7. Cardinalidade e valores dominantes


In [0]:
SELECT 'cod_material' AS coluna, COUNT(DISTINCT CAST(`cod_material` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'cod_centro' AS coluna, COUNT(DISTINCT CAST(`cod_centro` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'cod_empresa' AS coluna, COUNT(DISTINCT CAST(`cod_empresa` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'tp_suprimento' AS coluna, COUNT(DISTINCT CAST(`tp_suprimento` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'tp_grupo_determinacao_estoque' AS coluna, COUNT(DISTINCT CAST(`tp_grupo_determinacao_estoque` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'ind_remessa_jit' AS coluna, COUNT(DISTINCT CAST(`ind_remessa_jit` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'cod_chave_horizonte' AS coluna, COUNT(DISTINCT CAST(`cod_chave_horizonte` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'ind_entrada_lotes' AS coluna, COUNT(DISTINCT CAST(`ind_entrada_lotes` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'ind_coproduto' AS coluna, COUNT(DISTINCT CAST(`ind_coproduto` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'cod_deposito_suprimento_externo' AS coluna, COUNT(DISTINCT CAST(`cod_deposito_suprimento_externo` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'cod_deposito_producao' AS coluna, COUNT(DISTINCT CAST(`cod_deposito_producao` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'cod_calendario_planejamento' AS coluna, COUNT(DISTINCT CAST(`cod_calendario_planejamento` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'ind_baixa_explosao' AS coluna, COUNT(DISTINCT CAST(`ind_baixa_explosao` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'cod_perfil_cobertura' AS coluna, COUNT(DISTINCT CAST(`cod_perfil_cobertura` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'ind_material_granel' AS coluna, COUNT(DISTINCT CAST(`ind_material_granel` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'cod_margem_seguranca' AS coluna, COUNT(DISTINCT CAST(`cod_margem_seguranca` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'cod_perfil_periodo' AS coluna, COUNT(DISTINCT CAST(`cod_perfil_periodo` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'qt_margem_seguranca' AS coluna, COUNT(DISTINCT CAST(`qt_margem_seguranca` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'tp_suprimento_especial' AS coluna, COUNT(DISTINCT CAST(`tp_suprimento_especial` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'ind_utilizacao_quotizacao' AS coluna, COUNT(DISTINCT CAST(`ind_utilizacao_quotizacao` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'ind_aap' AS coluna, COUNT(DISTINCT CAST(`ind_aap` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'qt_tempo_producao_interna' AS coluna, COUNT(DISTINCT CAST(`qt_tempo_producao_interna` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'qt_estoque_seguranca' AS coluna, COUNT(DISTINCT CAST(`qt_estoque_seguranca` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'qt_estoque_minimo_seguranca' AS coluna, COUNT(DISTINCT CAST(`qt_estoque_minimo_seguranca` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'pct_grau_atendimento' AS coluna, COUNT(DISTINCT CAST(`pct_grau_atendimento` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'qt_dias_prazo_entrega_previsto' AS coluna, COUNT(DISTINCT CAST(`qt_dias_prazo_entrega_previsto` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'qt_dias_processamento_entrada' AS coluna, COUNT(DISTINCT CAST(`qt_dias_processamento_entrada` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'ind_marcado_exclusao' AS coluna, COUNT(DISTINCT CAST(`ind_marcado_exclusao` AS STRING)) AS distintos FROM tmp_zglmm159_m2_centro_4058
ORDER BY distintos ASC, coluna;


In [0]:
-- Domínio: cod_material
SELECT CAST(`cod_material` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m2_centro_4058
GROUP BY CAST(`cod_material` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: cod_centro
SELECT CAST(`cod_centro` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m2_centro_4058
GROUP BY CAST(`cod_centro` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: cod_empresa
SELECT CAST(`cod_empresa` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m2_centro_4058
GROUP BY CAST(`cod_empresa` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: tp_suprimento
SELECT CAST(`tp_suprimento` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m2_centro_4058
GROUP BY CAST(`tp_suprimento` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: tp_grupo_determinacao_estoque
SELECT CAST(`tp_grupo_determinacao_estoque` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m2_centro_4058
GROUP BY CAST(`tp_grupo_determinacao_estoque` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: ind_remessa_jit
SELECT CAST(`ind_remessa_jit` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m2_centro_4058
GROUP BY CAST(`ind_remessa_jit` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: cod_chave_horizonte
SELECT CAST(`cod_chave_horizonte` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m2_centro_4058
GROUP BY CAST(`cod_chave_horizonte` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: ind_entrada_lotes
SELECT CAST(`ind_entrada_lotes` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m2_centro_4058
GROUP BY CAST(`ind_entrada_lotes` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: ind_coproduto
SELECT CAST(`ind_coproduto` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m2_centro_4058
GROUP BY CAST(`ind_coproduto` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: cod_deposito_suprimento_externo
SELECT CAST(`cod_deposito_suprimento_externo` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m2_centro_4058
GROUP BY CAST(`cod_deposito_suprimento_externo` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: cod_deposito_producao
SELECT CAST(`cod_deposito_producao` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m2_centro_4058
GROUP BY CAST(`cod_deposito_producao` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: cod_calendario_planejamento
SELECT CAST(`cod_calendario_planejamento` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m2_centro_4058
GROUP BY CAST(`cod_calendario_planejamento` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: ind_baixa_explosao
SELECT CAST(`ind_baixa_explosao` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m2_centro_4058
GROUP BY CAST(`ind_baixa_explosao` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: cod_perfil_cobertura
SELECT CAST(`cod_perfil_cobertura` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m2_centro_4058
GROUP BY CAST(`cod_perfil_cobertura` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: ind_material_granel
SELECT CAST(`ind_material_granel` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m2_centro_4058
GROUP BY CAST(`ind_material_granel` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: cod_margem_seguranca
SELECT CAST(`cod_margem_seguranca` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m2_centro_4058
GROUP BY CAST(`cod_margem_seguranca` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: cod_perfil_periodo
SELECT CAST(`cod_perfil_periodo` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m2_centro_4058
GROUP BY CAST(`cod_perfil_periodo` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: tp_suprimento_especial
SELECT CAST(`tp_suprimento_especial` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m2_centro_4058
GROUP BY CAST(`tp_suprimento_especial` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: ind_utilizacao_quotizacao
SELECT CAST(`ind_utilizacao_quotizacao` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m2_centro_4058
GROUP BY CAST(`ind_utilizacao_quotizacao` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: ind_aap
SELECT CAST(`ind_aap` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m2_centro_4058
GROUP BY CAST(`ind_aap` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: ind_marcado_exclusao
SELECT CAST(`ind_marcado_exclusao` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m2_centro_4058
GROUP BY CAST(`ind_marcado_exclusao` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


## 8. Perfil numérico e extremos


In [0]:
SELECT 'qt_margem_seguranca' AS coluna,
MIN(`qt_margem_seguranca`) AS minimo,
MAX(`qt_margem_seguranca`) AS maximo,
AVG(`qt_margem_seguranca`) AS media,
SUM(`qt_margem_seguranca`) AS soma,
SUM(CASE WHEN `qt_margem_seguranca` < 0 THEN 1 ELSE 0 END) AS negativos,
SUM(CASE WHEN `qt_margem_seguranca` = 0 THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN `qt_margem_seguranca` > 0 THEN 1 ELSE 0 END) AS positivos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'qt_tempo_producao_interna' AS coluna,
MIN(`qt_tempo_producao_interna`) AS minimo,
MAX(`qt_tempo_producao_interna`) AS maximo,
AVG(`qt_tempo_producao_interna`) AS media,
SUM(`qt_tempo_producao_interna`) AS soma,
SUM(CASE WHEN `qt_tempo_producao_interna` < 0 THEN 1 ELSE 0 END) AS negativos,
SUM(CASE WHEN `qt_tempo_producao_interna` = 0 THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN `qt_tempo_producao_interna` > 0 THEN 1 ELSE 0 END) AS positivos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'qt_estoque_seguranca' AS coluna,
MIN(`qt_estoque_seguranca`) AS minimo,
MAX(`qt_estoque_seguranca`) AS maximo,
AVG(`qt_estoque_seguranca`) AS media,
SUM(`qt_estoque_seguranca`) AS soma,
SUM(CASE WHEN `qt_estoque_seguranca` < 0 THEN 1 ELSE 0 END) AS negativos,
SUM(CASE WHEN `qt_estoque_seguranca` = 0 THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN `qt_estoque_seguranca` > 0 THEN 1 ELSE 0 END) AS positivos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'qt_estoque_minimo_seguranca' AS coluna,
MIN(`qt_estoque_minimo_seguranca`) AS minimo,
MAX(`qt_estoque_minimo_seguranca`) AS maximo,
AVG(`qt_estoque_minimo_seguranca`) AS media,
SUM(`qt_estoque_minimo_seguranca`) AS soma,
SUM(CASE WHEN `qt_estoque_minimo_seguranca` < 0 THEN 1 ELSE 0 END) AS negativos,
SUM(CASE WHEN `qt_estoque_minimo_seguranca` = 0 THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN `qt_estoque_minimo_seguranca` > 0 THEN 1 ELSE 0 END) AS positivos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'pct_grau_atendimento' AS coluna,
MIN(`pct_grau_atendimento`) AS minimo,
MAX(`pct_grau_atendimento`) AS maximo,
AVG(`pct_grau_atendimento`) AS media,
SUM(`pct_grau_atendimento`) AS soma,
SUM(CASE WHEN `pct_grau_atendimento` < 0 THEN 1 ELSE 0 END) AS negativos,
SUM(CASE WHEN `pct_grau_atendimento` = 0 THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN `pct_grau_atendimento` > 0 THEN 1 ELSE 0 END) AS positivos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'qt_dias_prazo_entrega_previsto' AS coluna,
MIN(`qt_dias_prazo_entrega_previsto`) AS minimo,
MAX(`qt_dias_prazo_entrega_previsto`) AS maximo,
AVG(`qt_dias_prazo_entrega_previsto`) AS media,
SUM(`qt_dias_prazo_entrega_previsto`) AS soma,
SUM(CASE WHEN `qt_dias_prazo_entrega_previsto` < 0 THEN 1 ELSE 0 END) AS negativos,
SUM(CASE WHEN `qt_dias_prazo_entrega_previsto` = 0 THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN `qt_dias_prazo_entrega_previsto` > 0 THEN 1 ELSE 0 END) AS positivos
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'qt_dias_processamento_entrada' AS coluna,
MIN(`qt_dias_processamento_entrada`) AS minimo,
MAX(`qt_dias_processamento_entrada`) AS maximo,
AVG(`qt_dias_processamento_entrada`) AS media,
SUM(`qt_dias_processamento_entrada`) AS soma,
SUM(CASE WHEN `qt_dias_processamento_entrada` < 0 THEN 1 ELSE 0 END) AS negativos,
SUM(CASE WHEN `qt_dias_processamento_entrada` = 0 THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN `qt_dias_processamento_entrada` > 0 THEN 1 ELSE 0 END) AS positivos
FROM tmp_zglmm159_m2_centro_4058;


## 9. Datas, formatos e faixas


Não há campos com prefixo `dt_` no DESCRIBE recebido.


## 10. Zeros à esquerda e formato das chaves


In [0]:
SELECT 'cod_material' AS coluna,
MIN(LENGTH(TRIM(CAST(`cod_material` AS STRING)))) AS tamanho_min,
MAX(LENGTH(TRIM(CAST(`cod_material` AS STRING)))) AS tamanho_max,
SUM(CASE WHEN TRIM(CAST(`cod_material` AS STRING)) RLIKE '^0[0-9]+' THEN 1 ELSE 0 END) AS com_zero_esquerda,
COUNT(DISTINCT TRIM(CAST(`cod_material` AS STRING))) AS distintos_brutos,
COUNT(DISTINCT REGEXP_REPLACE(TRIM(CAST(`cod_material` AS STRING)), '^0+', '')) AS distintos_sem_zero
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'cod_centro' AS coluna,
MIN(LENGTH(TRIM(CAST(`cod_centro` AS STRING)))) AS tamanho_min,
MAX(LENGTH(TRIM(CAST(`cod_centro` AS STRING)))) AS tamanho_max,
SUM(CASE WHEN TRIM(CAST(`cod_centro` AS STRING)) RLIKE '^0[0-9]+' THEN 1 ELSE 0 END) AS com_zero_esquerda,
COUNT(DISTINCT TRIM(CAST(`cod_centro` AS STRING))) AS distintos_brutos,
COUNT(DISTINCT REGEXP_REPLACE(TRIM(CAST(`cod_centro` AS STRING)), '^0+', '')) AS distintos_sem_zero
FROM tmp_zglmm159_m2_centro_4058
UNION ALL
SELECT 'cod_empresa' AS coluna,
MIN(LENGTH(TRIM(CAST(`cod_empresa` AS STRING)))) AS tamanho_min,
MAX(LENGTH(TRIM(CAST(`cod_empresa` AS STRING)))) AS tamanho_max,
SUM(CASE WHEN TRIM(CAST(`cod_empresa` AS STRING)) RLIKE '^0[0-9]+' THEN 1 ELSE 0 END) AS com_zero_esquerda,
COUNT(DISTINCT TRIM(CAST(`cod_empresa` AS STRING))) AS distintos_brutos,
COUNT(DISTINCT REGEXP_REPLACE(TRIM(CAST(`cod_empresa` AS STRING)), '^0+', '')) AS distintos_sem_zero
FROM tmp_zglmm159_m2_centro_4058;


## 11. Linhas integralmente duplicadas


In [0]:
SELECT
  COUNT(*) AS linhas,
  COUNT(DISTINCT SHA2(CONCAT_WS('§', COALESCE(CAST(`cod_material` AS STRING), '<NULL>'), COALESCE(CAST(`cod_centro` AS STRING), '<NULL>'), COALESCE(CAST(`cod_empresa` AS STRING), '<NULL>'), COALESCE(CAST(`tp_suprimento` AS STRING), '<NULL>'), COALESCE(CAST(`tp_grupo_determinacao_estoque` AS STRING), '<NULL>'), COALESCE(CAST(`ind_remessa_jit` AS STRING), '<NULL>'), COALESCE(CAST(`cod_chave_horizonte` AS STRING), '<NULL>'), COALESCE(CAST(`ind_entrada_lotes` AS STRING), '<NULL>'), COALESCE(CAST(`ind_coproduto` AS STRING), '<NULL>'), COALESCE(CAST(`cod_deposito_suprimento_externo` AS STRING), '<NULL>'), COALESCE(CAST(`cod_deposito_producao` AS STRING), '<NULL>'), COALESCE(CAST(`cod_calendario_planejamento` AS STRING), '<NULL>'), COALESCE(CAST(`ind_baixa_explosao` AS STRING), '<NULL>'), COALESCE(CAST(`cod_perfil_cobertura` AS STRING), '<NULL>'), COALESCE(CAST(`ind_material_granel` AS STRING), '<NULL>'), COALESCE(CAST(`cod_margem_seguranca` AS STRING), '<NULL>'), COALESCE(CAST(`cod_perfil_periodo` AS STRING), '<NULL>'), COALESCE(CAST(`qt_margem_seguranca` AS STRING), '<NULL>'), COALESCE(CAST(`tp_suprimento_especial` AS STRING), '<NULL>'), COALESCE(CAST(`ind_utilizacao_quotizacao` AS STRING), '<NULL>'), COALESCE(CAST(`ind_aap` AS STRING), '<NULL>'), COALESCE(CAST(`qt_tempo_producao_interna` AS STRING), '<NULL>'), COALESCE(CAST(`qt_estoque_seguranca` AS STRING), '<NULL>'), COALESCE(CAST(`qt_estoque_minimo_seguranca` AS STRING), '<NULL>'), COALESCE(CAST(`pct_grau_atendimento` AS STRING), '<NULL>'), COALESCE(CAST(`qt_dias_prazo_entrega_previsto` AS STRING), '<NULL>'), COALESCE(CAST(`qt_dias_processamento_entrada` AS STRING), '<NULL>'), COALESCE(CAST(`ind_marcado_exclusao` AS STRING), '<NULL>')), 256)) AS linhas_distintas,
  COUNT(*) - COUNT(DISTINCT SHA2(CONCAT_WS('§', COALESCE(CAST(`cod_material` AS STRING), '<NULL>'), COALESCE(CAST(`cod_centro` AS STRING), '<NULL>'), COALESCE(CAST(`cod_empresa` AS STRING), '<NULL>'), COALESCE(CAST(`tp_suprimento` AS STRING), '<NULL>'), COALESCE(CAST(`tp_grupo_determinacao_estoque` AS STRING), '<NULL>'), COALESCE(CAST(`ind_remessa_jit` AS STRING), '<NULL>'), COALESCE(CAST(`cod_chave_horizonte` AS STRING), '<NULL>'), COALESCE(CAST(`ind_entrada_lotes` AS STRING), '<NULL>'), COALESCE(CAST(`ind_coproduto` AS STRING), '<NULL>'), COALESCE(CAST(`cod_deposito_suprimento_externo` AS STRING), '<NULL>'), COALESCE(CAST(`cod_deposito_producao` AS STRING), '<NULL>'), COALESCE(CAST(`cod_calendario_planejamento` AS STRING), '<NULL>'), COALESCE(CAST(`ind_baixa_explosao` AS STRING), '<NULL>'), COALESCE(CAST(`cod_perfil_cobertura` AS STRING), '<NULL>'), COALESCE(CAST(`ind_material_granel` AS STRING), '<NULL>'), COALESCE(CAST(`cod_margem_seguranca` AS STRING), '<NULL>'), COALESCE(CAST(`cod_perfil_periodo` AS STRING), '<NULL>'), COALESCE(CAST(`qt_margem_seguranca` AS STRING), '<NULL>'), COALESCE(CAST(`tp_suprimento_especial` AS STRING), '<NULL>'), COALESCE(CAST(`ind_utilizacao_quotizacao` AS STRING), '<NULL>'), COALESCE(CAST(`ind_aap` AS STRING), '<NULL>'), COALESCE(CAST(`qt_tempo_producao_interna` AS STRING), '<NULL>'), COALESCE(CAST(`qt_estoque_seguranca` AS STRING), '<NULL>'), COALESCE(CAST(`qt_estoque_minimo_seguranca` AS STRING), '<NULL>'), COALESCE(CAST(`pct_grau_atendimento` AS STRING), '<NULL>'), COALESCE(CAST(`qt_dias_prazo_entrega_previsto` AS STRING), '<NULL>'), COALESCE(CAST(`qt_dias_processamento_entrada` AS STRING), '<NULL>'), COALESCE(CAST(`ind_marcado_exclusao` AS STRING), '<NULL>')), 256)) AS duplicatas_integrais
FROM tmp_zglmm159_m2_centro_4058;


## 12. Amostra ampliada — validar formatos reais


In [0]:
SELECT * FROM tmp_zglmm159_m2_centro_4058 ORDER BY cod_centro, cod_material LIMIT 100;


## 13. Resumo automático de cenário e chave


In [0]:
WITH base AS (
  SELECT COUNT(*) AS linhas,
         COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_centro` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_empresa` AS STRING)), '<NULL>'))) AS chaves
  FROM tmp_zglmm159_m2_centro_4058
)
SELECT 'CENARIO' AS bloco, 'tabela' AS item, 'tmp_zglmm159_m2_centro_4058' AS valor
UNION ALL SELECT 'CENARIO', 'colunas', '28'
UNION ALL SELECT 'CENARIO', 'linhas', CAST(linhas AS STRING) FROM base
UNION ALL SELECT 'CHAVE CANDIDATA', 'campos', 'cod_material + cod_centro + cod_empresa'
UNION ALL SELECT 'CHAVE CANDIDATA', 'distintos', CAST(chaves AS STRING) FROM base
UNION ALL SELECT 'CHAVE CANDIDATA', 'veredito', CASE WHEN linhas = chaves THEN 'CHAVE UNICA' ELSE 'NAO UNICA - INVESTIGAR DIMENSAO ADICIONAL' END FROM base;


---
## Próximo passo

1. Revisar a granularidade real e a chave candidata.
2. Confirmar se as colunas documentadas como sem fonte permanecem realmente nulas.
3. Escolher um recorte reproduzível para extração SAP, preservando todas as visões C1, M1 e M2.
4. Não avançar para o de/para antes de validar o diagnóstico desta análise exploratória.


## Controle específico do cenário anterior

A validação anterior do centro 4058 tratou M2 como 1 linha por material. O notebook testa se empresa ou tipo de suprimento especial criam granularidade adicional na nova view.


In [0]:
SELECT
  cod_material,
  COUNT(*) AS linhas,
  COUNT(DISTINCT cod_empresa) AS empresas,
  COUNT(DISTINCT tp_suprimento_especial) AS tipos_suprimento_especial
FROM tmp_zglmm159_m2_centro_4058
GROUP BY cod_material
HAVING COUNT(*) > 1
ORDER BY linhas DESC, cod_material;


In [0]:
SELECT *
FROM dev_procurement.corp_curated.vw_zglmm159_m2
WHERE TRIM(CAST(cod_centro AS STRING)) = '4058';

## Portão do cenário 4058

Antes de extrair novamente o SAP, confirme:
- a volumetria da nova visão para o centro;
- a chave e a granularidade real;
- a preservação dos múltiplos depósitos na M1;
- o preenchimento real das colunas documentadas como sem fonte;
- a existência de valores ou dimensões inesperadas.
